In [0]:
CREATE OR REPLACE TABLE customers_cdc AS
SELECT * FROM VALUES
    (1, 'Ali', 'ali@example.com', 'Tashkent'),
    (2, 'Bobur', 'bobur@example.com', 'Samarkand'),
    (3, 'John', 'john@example.com', 'Bukhara'),
    (4, 'Sarah', 'sarah@example.com', 'Andijan')
AS t(customer_id, name, email, city);


CREATE OR REPLACE TABLE customer_changes AS
SELECT * FROM VALUES
    (2, 'Bobur', 'bobur.updated@example.com', 'Tashkent', 'UPDATE'),
    (3, 'John', 'john@example.com', 'Bukhara', 'DELETE'),
    (5, 'David', 'david@example.com', 'Namangan', 'INSERT'),
    (6, 'Mary', 'mary@example.com', 'Tashkent', 'INSERT')
AS t(customer_id, name, email, city, operation);


MERGE INTO customers_cdc AS target
USING customer_changes AS source
ON target.customer_id = source.customer_id

WHEN MATCHED AND source.operation = 'UPDATE' THEN
    UPDATE SET
        target.name = source.name,
        target.email = source.email,
        target.city = source.city

WHEN MATCHED AND source.operation = 'DELETE' THEN
    DELETE

WHEN NOT MATCHED AND source.operation = 'INSERT' THEN
    INSERT (customer_id, name, email, city)
    VALUES (source.customer_id, source.name, source.email, source.city)
;


SELECT * FROM customers_cdc;

SELECT * FROM customer_changes;